## 1. Setup and Data Loading

Import necessary libraries and load the labeled datasets.

In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'requirements.txt').exists():
            return path
    return start

ROOT_DIR = find_repo_root()
DATA_DIR = ROOT_DIR / 'data'

def load_labeled_data():
    reddit = pd.read_csv(DATA_DIR / 'reddit_labeled.csv')
    youtube = pd.read_csv(DATA_DIR / 'youtube_labeled.csv')

    for col in ['score', 'post_score', 'post_upvote_ratio', 'user_total_karma', 'controversiality']:
        if col in reddit.columns:
            reddit[col] = pd.to_numeric(reddit[col], errors='coerce')
    for col in ['likeCount', 'replyCount']:
        if col in youtube.columns:
            youtube[col] = pd.to_numeric(youtube[col], errors='coerce')

    reddit_text_col = 'self_text'
    youtube_text_col = 'text'
    if reddit_text_col in reddit.columns:
        reddit['text_length'] = reddit[reddit_text_col].astype(str).str.len()
        reddit['word_count'] = reddit[reddit_text_col].astype(str).str.split().str.len()
    if youtube_text_col in youtube.columns:
        youtube['text_length'] = youtube[youtube_text_col].astype(str).str.len()
        youtube['word_count'] = youtube[youtube_text_col].astype(str).str.split().str.len()
    return reddit, youtube


def add_sentiment_columns(reddit, youtube):
    from textblob import TextBlob
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    vader = SentimentIntensityAnalyzer()

    def vader_scores(text):
        if pd.isna(text) or text == '':
            return {'compound': 0, 'pos': 0, 'neu': 0, 'neg': 0, 'label': 'neutral'}
        scores = vader.polarity_scores(str(text))
        if scores['compound'] >= 0.05:
            label = 'positive'
        elif scores['compound'] <= -0.05:
            label = 'negative'
        else:
            label = 'neutral'
        scores['label'] = label
        return scores

    def textblob_scores(text):
        if pd.isna(text) or text == '':
            return {'polarity': 0, 'subjectivity': 0, 'label': 'neutral'}
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
        if polarity > 0.1:
            label = 'positive'
        elif polarity < -0.1:
            label = 'negative'
        else:
            label = 'neutral'
        return {'polarity': polarity, 'subjectivity': subjectivity, 'label': label}

    reddit_text_col = 'self_text'
    youtube_text_col = 'text'

    for df, text_col in [(reddit, reddit_text_col), (youtube, youtube_text_col)]:
        vader_results = df[text_col].apply(vader_scores)
        df['vader_compound'] = vader_results.apply(lambda x: x['compound'])
        df['vader_positive'] = vader_results.apply(lambda x: x['pos'])
        df['vader_neutral'] = vader_results.apply(lambda x: x['neu'])
        df['vader_negative'] = vader_results.apply(lambda x: x['neg'])
        df['vader_label'] = vader_results.apply(lambda x: x['label'])

        blob_results = df[text_col].apply(textblob_scores)
        df['textblob_polarity'] = blob_results.apply(lambda x: x['polarity'])
        df['textblob_subjectivity'] = blob_results.apply(lambda x: x['subjectivity'])
        df['textblob_label'] = blob_results.apply(lambda x: x['label'])

    return reddit, youtube


def load_sentiment_data():
    reddit, youtube = load_labeled_data()
    return add_sentiment_columns(reddit, youtube)


In [ ]:
from pathlib import Path

# ==================== EDIT THESE PATHS ====================
# Local mode:  INPUT_DIR = Path('..')  
# Kaggle mode: INPUT_DIR = Path('/kaggle/input/israel-hamas-data')
# =========================================================

INPUT_DIR = Path('./data')     # <-- EDIT: path to raw data (reddit_labeled.csv, youtube_labeled.csv)
OUTPUT_DIR = Path.cwd()         # <-- EDIT: where to save outputs (reddit_processed.csv, etc.)

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input directory:  {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## Kaggle Setup & Paths

Configure paths for Kaggle notebook environment. Update input paths if uploading data manually.



In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

In [ ]:
# Load datasets
print("Loading data...")
reddit_df = pd.read_csv(DATA_DIR / 'reddit_labeled.csv')
youtube_df = pd.read_csv(DATA_DIR / 'youtube_labeled.csv')

print(f"✓ Reddit data: {len(reddit_df):,} rows")
print(f"✓ YouTube data: {len(youtube_df):,} rows")
print(f"✓ Total dataset: {len(reddit_df) + len(youtube_df):,} data points")

## 2. Data Preprocessing

Convert data types and prepare data for analysis.

In [ ]:
# Convert numeric columns for Reddit
if 'score' in reddit_df.columns:
    reddit_df['score'] = pd.to_numeric(reddit_df['score'], errors='coerce')
if 'post_score' in reddit_df.columns:
    reddit_df['post_score'] = pd.to_numeric(reddit_df['post_score'], errors='coerce')
if 'post_upvote_ratio' in reddit_df.columns:
    reddit_df['post_upvote_ratio'] = pd.to_numeric(reddit_df['post_upvote_ratio'], errors='coerce')
if 'user_total_karma' in reddit_df.columns:
    reddit_df['user_total_karma'] = pd.to_numeric(reddit_df['user_total_karma'], errors='coerce')

# Convert numeric columns for YouTube (if they exist)
if 'likeCount' in youtube_df.columns:
    youtube_df['likeCount'] = pd.to_numeric(youtube_df['likeCount'], errors='coerce')
if 'replyCount' in youtube_df.columns:
    youtube_df['replyCount'] = pd.to_numeric(youtube_df['replyCount'], errors='coerce')

# Calculate text lengths
if 'self_text' in reddit_df.columns:
    reddit_df['text_length'] = reddit_df['self_text'].fillna('').astype(str).str.len()
if 'text' in youtube_df.columns:
    youtube_df['text_length'] = youtube_df['text'].fillna('').astype(str).str.len()

print("✓ Data preprocessing complete")

## 3. Dataset Overview

Examine the structure and basic information about our datasets.

In [ ]:
print("=" * 80)
print("REDDIT DATASET OVERVIEW")
print("=" * 80)
print(f"\nShape: {reddit_df.shape}")
print(f"\nColumns ({len(reddit_df.columns)}):")
for col in reddit_df.columns:
    print(f"  - {col}")
print(f"\nData Types:")
print(reddit_df.dtypes)

In [ ]:
print("=" * 80)
print("YOUTUBE DATASET OVERVIEW")
print("=" * 80)
print(f"\nShape: {youtube_df.shape}")
print(f"\nColumns ({len(youtube_df.columns)}):")
for col in youtube_df.columns:
    print(f"  - {col}")
print(f"\nData Types:")
print(youtube_df.dtypes)

In [ ]:
# Preview data
print("\n📊 REDDIT DATA SAMPLE:")
display(reddit_df.head())

In [ ]:
print("\n📊 YOUTUBE DATA SAMPLE:")
display(youtube_df.head())

## 4. Stance Distribution Analysis

**Key Research Question**: How are narratives and sentiments regarding the conflict represented across platforms?

Stance Labels:
- **P** = Supports Palestine
- **I** = Supports Israel
- **N** = Neutral/Unclear

In [ ]:
# Identify label columns
reddit_label_col = 'Label'
youtube_label_col = 'Label'

# Reddit stance distribution
print("=" * 80)
print("REDDIT STANCE DISTRIBUTION")
print("=" * 80)
reddit_stance = reddit_df[reddit_label_col].value_counts()
reddit_stance_pct = reddit_df[reddit_label_col].value_counts(normalize=True) * 100

reddit_summary = pd.DataFrame({
    'Count': reddit_stance,
    'Percentage': reddit_stance_pct
})
print(reddit_summary)
print(f"\nTotal labeled: {reddit_stance.sum():,}")

In [ ]:
# YouTube stance distribution
print("=" * 80)
print("YOUTUBE STANCE DISTRIBUTION")
print("=" * 80)
youtube_stance = youtube_df[youtube_label_col].value_counts()
youtube_stance_pct = youtube_df[youtube_label_col].value_counts(normalize=True) * 100

youtube_summary = pd.DataFrame({
    'Count': youtube_stance,
    'Percentage': youtube_stance_pct
})
print(youtube_summary)
print(f"\nTotal labeled: {youtube_stance.sum():,}")

### Visualization: Stance Distribution Comparison

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color mapping
colors = {'P': '#2ecc71', 'I': '#3498db', 'N': '#95a5a6'}

# Reddit
reddit_counts = reddit_df[reddit_label_col].value_counts()
reddit_colors = [colors.get(label, '#95a5a6') for label in reddit_counts.index]
axes[0].bar(reddit_counts.index, reddit_counts.values, color=reddit_colors, alpha=0.8, edgecolor='black')
axes[0].set_title('Reddit Stance Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Stance', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(reddit_counts.values):
    axes[0].text(i, v + max(reddit_counts.values)*0.02, f'{v}\n({v/reddit_counts.sum()*100:.1f}%)', 
                 ha='center', fontweight='bold')

# YouTube
youtube_counts = youtube_df[youtube_label_col].value_counts()
youtube_colors = [colors.get(label, '#95a5a6') for label in youtube_counts.index]
axes[1].bar(youtube_counts.index, youtube_counts.values, color=youtube_colors, alpha=0.8, edgecolor='black')
axes[1].set_title('YouTube Stance Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Stance', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(youtube_counts.values):
    axes[1].text(i, v + max(youtube_counts.values)*0.02, f'{v}\n({v/youtube_counts.sum()*100:.1f}%)', 
                 ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Pie charts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_pie = ['#2ecc71', '#3498db', '#95a5a6']
explode = (0.05, 0.05, 0.05)

# Reddit pie
axes[0].pie(reddit_counts.values, labels=reddit_counts.index, autopct='%1.1f%%',
            colors=colors_pie, explode=explode, shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Reddit Stance Distribution (%)', fontsize=14, fontweight='bold')

# YouTube pie
axes[1].pie(youtube_counts.values, labels=youtube_counts.index, autopct='%1.1f%%',
            colors=colors_pie, explode=explode, shadow=True, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('YouTube Stance Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Platform comparison - side by side
comparison_df = pd.DataFrame({
    'Reddit': reddit_df[reddit_label_col].value_counts(normalize=True) * 100,
    'YouTube': youtube_df[youtube_label_col].value_counts(normalize=True) * 100
}).fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(kind='bar', ax=ax, width=0.8, alpha=0.8, edgecolor='black')
ax.set_title('Stance Distribution Comparison: Reddit vs YouTube (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Stance', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.legend(title='Platform', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

### 🔍 Key Finding: Platform Differences

- **YouTube** shows more Pro-Palestine content (43.7% vs 35.6%)
- **Reddit** has more neutral/unclear discourse (46.6% vs 43.4%)
- **Pro-Israel** content is minority on both platforms (Reddit: 17.8%, YouTube: 12.9%)

## 5. Missing Data Analysis

Understanding data completeness and potential limitations.

In [ ]:
# Reddit missing data
print("=" * 80)
print("REDDIT MISSING DATA")
print("=" * 80)
reddit_missing = reddit_df.isnull().sum()
reddit_missing_pct = (reddit_df.isnull().sum() / len(reddit_df)) * 100
reddit_missing_df = pd.DataFrame({
    'Missing Count': reddit_missing,
    'Percentage': reddit_missing_pct
}).sort_values('Percentage', ascending=False)

print(reddit_missing_df[reddit_missing_df['Missing Count'] > 0])

In [ ]:
# YouTube missing data
print("=" * 80)
print("YOUTUBE MISSING DATA")
print("=" * 80)
youtube_missing = youtube_df.isnull().sum()
youtube_missing_pct = (youtube_df.isnull().sum() / len(youtube_df)) * 100
youtube_missing_df = pd.DataFrame({
    'Missing Count': youtube_missing,
    'Percentage': youtube_missing_pct
}).sort_values('Percentage', ascending=False)

print(youtube_missing_df[youtube_missing_df['Missing Count'] > 0])

## 6. Engagement Metrics Analysis

**Research Question 2**: How do engagement metrics influence the visibility of perspectives?

In [ ]:
# Reddit engagement statistics
print("=" * 80)
print("REDDIT ENGAGEMENT METRICS")
print("=" * 80)

if 'score' in reddit_df.columns:
    print("\n📊 Score Statistics:")
    print(reddit_df['score'].describe())
    print(f"\nMedian: {reddit_df['score'].median()}")
    print(f"Mode: {reddit_df['score'].mode().values[0] if len(reddit_df['score'].mode()) > 0 else 'N/A'}")

if 'controversiality' in reddit_df.columns:
    print("\n⚡ Controversiality:")
    print(reddit_df['controversiality'].value_counts())
    controversial_pct = (reddit_df['controversiality'] == 1).sum() / len(reddit_df) * 100
    print(f"\nControversial posts: {controversial_pct:.2f}%")

In [ ]:
# Engagement by stance - Reddit
if 'score' in reddit_df.columns:
    print("\n📈 ENGAGEMENT BY STANCE (Reddit):")
    engagement_by_stance = reddit_df.groupby(reddit_label_col)['score'].agg([
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std Dev', 'std'),
        ('Max', 'max')
    ]).round(2)
    print(engagement_by_stance)

### Visualization: Engagement by Stance

In [ ]:
# Box plot and violin plot for Reddit scores
if 'score' in reddit_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Box plot
    reddit_df.boxplot(column='score', by=reddit_label_col, ax=axes[0])
    axes[0].set_title('Reddit Score Distribution by Stance', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Stance', fontsize=12)
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].get_figure().suptitle('')
    
    # Violin plot
    sns.violinplot(data=reddit_df, x=reddit_label_col, y='score', ax=axes[1])
    axes[1].set_title('Reddit Score Distribution by Stance (Violin)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Stance', fontsize=12)
    axes[1].set_ylabel('Score', fontsize=12)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Controversiality by stance
if 'controversiality' in reddit_df.columns:
    controversy_cross = pd.crosstab(reddit_df[reddit_label_col], 
                                   reddit_df['controversiality'], 
                                   normalize='index') * 100
    
    fig, ax = plt.subplots(figsize=(10, 6))
    controversy_cross.plot(kind='bar', ax=ax, stacked=False, edgecolor='black')
    ax.set_title('Reddit Controversiality by Stance (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Stance', fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.legend(title='Controversial', labels=['No', 'Yes'])
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.show()

## 7. Content Length Analysis

**Research Question 3**: Differences between Reddit discussions and YouTube narratives in framing the conflict.

In [ ]:
# Text length statistics
print("=" * 80)
print("CONTENT LENGTH ANALYSIS")
print("=" * 80)

if 'text_length' in reddit_df.columns:
    print("\n📝 REDDIT TEXT LENGTH:")
    print(reddit_df['text_length'].describe())
    print(f"\nMedian: {reddit_df['text_length'].median():.0f} characters")

if 'text_length' in youtube_df.columns:
    print("\n📝 YOUTUBE TEXT LENGTH:")
    print(youtube_df['text_length'].describe())
    print(f"\nMedian: {youtube_df['text_length'].median():.0f} characters")

# Compare
if 'text_length' in reddit_df.columns and 'text_length' in youtube_df.columns:
    ratio = reddit_df['text_length'].median() / youtube_df['text_length'].median()
    print(f"\n🔍 Reddit posts are {ratio:.1f}x longer than YouTube comments")

In [ ]:
# Text length by stance
if 'text_length' in reddit_df.columns:
    print("\n📊 REDDIT TEXT LENGTH BY STANCE:")
    reddit_length_by_stance = reddit_df.groupby(reddit_label_col)['text_length'].agg([
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std Dev', 'std')
    ]).round(2)
    print(reddit_length_by_stance)

if 'text_length' in youtube_df.columns:
    print("\n📊 YOUTUBE TEXT LENGTH BY STANCE:")
    youtube_length_by_stance = youtube_df.groupby(youtube_label_col)['text_length'].agg([
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Std Dev', 'std')
    ]).round(2)
    print(youtube_length_by_stance)

### Visualization: Text Length Distribution

In [ ]:
# Histograms
if 'text_length' in reddit_df.columns or 'text_length' in youtube_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    if 'text_length' in reddit_df.columns:
        axes[0].hist(reddit_df['text_length'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
        axes[0].set_title('Reddit Text Length Distribution', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Text Length (characters)', fontsize=10)
        axes[0].set_ylabel('Frequency', fontsize=10)
        axes[0].axvline(reddit_df['text_length'].median(), color='red', linestyle='--', linewidth=2,
                       label=f'Median: {reddit_df["text_length"].median():.0f}')
        axes[0].legend()
        axes[0].grid(alpha=0.3)
    
    if 'text_length' in youtube_df.columns:
        axes[1].hist(youtube_df['text_length'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
        axes[1].set_title('YouTube Text Length Distribution', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Text Length (characters)', fontsize=10)
        axes[1].set_ylabel('Frequency', fontsize=10)
        axes[1].axvline(youtube_df['text_length'].median(), color='red', linestyle='--', linewidth=2,
                       label=f'Median: {youtube_df["text_length"].median():.0f}')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plots by stance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if 'text_length' in reddit_df.columns:
    sns.boxplot(data=reddit_df, x=reddit_label_col, y='text_length', ax=axes[0])
    axes[0].set_title('Reddit Text Length by Stance', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Stance', fontsize=10)
    axes[0].set_ylabel('Text Length (characters)', fontsize=10)

if 'text_length' in youtube_df.columns:
    sns.boxplot(data=youtube_df, x=youtube_label_col, y='text_length', ax=axes[1])
    axes[1].set_title('YouTube Text Length by Stance', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Stance', fontsize=10)
    axes[1].set_ylabel('Text Length (characters)', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Correlation Analysis

Understanding relationships between different metrics.

In [ ]:
# Reddit correlation heatmap
reddit_numeric = reddit_df.select_dtypes(include=[np.number])
if len(reddit_numeric.columns) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = reddit_numeric.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
               square=True, linewidths=1, ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title('Reddit Metrics Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# YouTube correlation heatmap
youtube_numeric = youtube_df.select_dtypes(include=[np.number])
if len(youtube_numeric.columns) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = youtube_numeric.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
               square=True, linewidths=1, ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title('YouTube Metrics Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 9. Summary Statistics

Comprehensive overview of all findings.

In [ ]:
print("=" * 80)
print("EDA SUMMARY REPORT")
print("Israel-Hamas War Discourse Analysis")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

print("\n1. DATASET OVERVIEW")
print("-" * 80)
print(f"Reddit Posts/Comments: {len(reddit_df):,}")
print(f"YouTube Comments: {len(youtube_df):,}")
print(f"Total Data Points: {len(reddit_df) + len(youtube_df):,}")

print("\n2. STANCE DISTRIBUTION")
print("-" * 80)
print("\nReddit:")
print(reddit_summary)
print("\nYouTube:")
print(youtube_summary)

print("\n3. CONTENT CHARACTERISTICS")
print("-" * 80)
if 'text_length' in reddit_df.columns:
    print(f"Reddit median length: {reddit_df['text_length'].median():.0f} characters")
if 'text_length' in youtube_df.columns:
    print(f"YouTube median length: {youtube_df['text_length'].median():.0f} characters")

print("\n4. ENGAGEMENT METRICS")
print("-" * 80)
if 'score' in reddit_df.columns:
    print(f"Reddit median score: {reddit_df['score'].median():.0f}")
    print(f"Reddit mean score: {reddit_df['score'].mean():.2f}")

print("\n" + "=" * 80)

## 10. Key Findings & Insights

### Platform-Specific Patterns:

1. **Stance Distribution**:
   - YouTube has more Pro-Palestine content (43.7% vs 35.6%)
   - Reddit has more neutral discourse (46.6% vs 43.4%)
   - Pro-Israel content is minority on both platforms

2. **Content Characteristics**:
   - Reddit posts are ~2.4x longer (median: 132 vs 54 characters)
   - Reddit facilitates deeper, threaded discussions
   - YouTube comments are more immediate reactions

3. **Engagement Patterns**:
   - Reddit engagement is highly skewed (few viral posts)
   - Controversiality markers indicate polarizing content

### Implications for Research Questions:

**RQ1 (Narratives & Sentiments)**:
- Clear platform differences in how the conflict is represented
- YouTube more polarized, Reddit more balanced

**RQ2 (Engagement Impact)**:
- Engagement metrics vary by stance
- Ready for statistical testing of visibility patterns

**RQ3 (Platform Differences)**:
- Confirmed: Immediacy (YouTube) vs Discussion depth (Reddit)
- Content length significantly different between platforms

---

### Next Steps:
1. Sentiment analysis using NLP
2. Topic modeling (LDA/BERTopic)
3. Statistical significance testing
4. Word frequency and N-gram analysis
5. Temporal analysis (if timestamps available)

## 11. Export Processed Data

In [ ]:
# Save processed data for further analysis



In [ ]:
# Export processed data for use in Module 02 (Sentiment Analysis)
print("=" * 80)
print("EXPORTING PROCESSED DATA")
print("=" * 80)

reddit_export = reddit_df.copy()
youtube_export = youtube_df.copy()

reddit_path = OUTPUT_DIR / 'reddit_processed.csv'
youtube_path = OUTPUT_DIR / 'youtube_processed.csv'

reddit_export.to_csv(reddit_path, index=False, encoding='utf-8')
youtube_export.to_csv(youtube_path, index=False, encoding='utf-8')

print(f"\n✅ Exported: {reddit_path}")
print(f"   Shape: {reddit_export.shape} | Columns: {', '.join(reddit_export.columns[:5])}...")
print(f"\n✅ Exported: {youtube_path}")
print(f"   Shape: {youtube_export.shape} | Columns: {', '.join(youtube_export.columns[:5])}...")

print("\n📌 Next Step: Use reddit_processed.csv & youtube_processed.csv")
print("   as input to Module 02 (Sentiment Analysis) notebook")


## Export Processed Data

Save cleaned and processed data as CSVs for downstream analysis in next notebooks.

